In [ ]:




# i have used 3-week/synthetic-clustering-dataset.csv as for week 2 it was missing .
# All the answers in the notebook are below the code cell.

import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

In [4]:
df = pd.read_csv("../datasets/synthetic-clustering-dataset.csv")

df.head()

,student_key,course_week,group_code,quiz_completed,quiz_score_pct,task_submitted,submission_delay_minutes,task_attempts,monday_activity_count,recorded_attendance,concept_map_submitted,concept_coverage_pct,artifact_size_kb,previous_task_score_pct
0,SYN001,1,G07,1,98.3,1,6.0,3,29,1,0,NaN,1126.4,75.1
1,SYN001,2,G07,1,94.6,1,7.2,4,36,1,1,81.8,875.0,66.0
2,SYN001,3,G07,1,85.4,1,-11.8,5,35,1,1,90.5,775.5,87.9
3,SYN002,1,G03,1,66.8,1,-27.4,1,24,1,1,57.2,210.3,77.2
4,SYN002,2,G03,1,69.0,0,NaN,0,36,1,1,52.8,190.7,65.0


In [6]:
# Target y: did the student complete the quiz?
y = df["quiz_completed"]

# Features X: information used for prediction
features = [
    "course_week",
    "previous_task_score_pct",
    "task_attempts",
    "monday_activity_count"
]

X = df[features]

In [ ]:
# Checked  target-class imbalance
print(y.value_counts())
print()
print(y.value_counts(normalize=True))

quiz_completed
1    92
0    28
Name: count, dtype: int64

quiz_completed
1    0.766667
0    0.233333
Name: proportion, dtype: float64


In [8]:
# Same split will be used for every model
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

In [9]:
# Baseline: always predicts the most common class
baseline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", DummyClassifier(strategy="most_frequent"))
])

baseline.fit(X_train, y_train)
baseline_predictions = baseline.predict(X_test)

print("Baseline accuracy:", accuracy_score(y_test, baseline_predictions))
print("Confusion matrix:")
print(confusion_matrix(y_test, baseline_predictions))
print(classification_report(y_test, baseline_predictions, zero_division=0))

Baseline accuracy: 0.7666666666666667
Confusion matrix:
[[ 0  7]
 [ 0 23]]
              precision    recall  f1-score   support

           0       0.00      0.00      0.00         7
           1       0.77      1.00      0.87        23

    accuracy                           0.77        30
   macro avg       0.38      0.50      0.43        30
weighted avg       0.59      0.77      0.67        30



In [10]:
# First real model
logistic_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", LogisticRegression(max_iter=1000))
])

logistic_model.fit(X_train, y_train)
logistic_predictions = logistic_model.predict(X_test)

print("Logistic Regression accuracy:",
      accuracy_score(y_test, logistic_predictions))
print(confusion_matrix(y_test, logistic_predictions))

Logistic Regression accuracy: 0.7333333333333333
[[ 4  3]
 [ 5 18]]


In [11]:
# Second real model
tree_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", DecisionTreeClassifier(max_depth=3, random_state=42))
])

tree_model.fit(X_train, y_train)
tree_predictions = tree_model.predict(X_test)

print("Decision Tree accuracy:",
      accuracy_score(y_test, tree_predictions))
print(confusion_matrix(y_test, tree_predictions))

Decision Tree accuracy: 0.8
[[ 4  3]
 [ 3 20]]


## 1. Question

Why did the Decision Tree give better accuracy than Logistic Regression on the same dataset?

## 2. Investigation

I used the same target, features, train/test split and median imputation for both models. I compared their test accuracy and confusion matrices.

## 3. Evidence

Logistic Regression accuracy was 73.3%.

Confusion matrix:

[[4, 3],
 [5, 18]]

Decision Tree accuracy was 80.0%.

Confusion matrix:

[[4, 3],
 [3, 20]]

The Decision Tree made two fewer mistakes. It correctly predicted 20 completed quizzes, while Logistic Regression correctly predicted 18.

## 4. Conclusion

The Decision Tree may handle threshold-based or non-linear patterns in the data better than Logistic Regression. However, the test set had only 30 rows and the difference was only two predictions. We cannot yet conclude that the Decision Tree is clearly the better model.